In [1]:
(* 
    Olaf Surgut, Version 1.5.0 
*)

let ( let* ) xs ys = List.concat_map ys xs

let create_list n v = List.init n (fun _ -> v)

let build_row ps n =
    let rec help ps n =
        match ps, n with
        | _,  n when n < 0 ->
            []
        | [], n -> 
            [create_list n false]
        | p :: ps', _ ->
            let take = help ps' (n - 1 - p) in
            let dont = help ps  (n - 1) in
            (List.map (fun xs -> false :: ((create_list p true) @ xs)) take) @
            (List.map (fun xs -> false :: xs) dont)
    in
    List.map (
        fun xs -> List.tl xs
    ) (help ps (n + 1))

let rec build_candidate pss n =
    match pss with
    | [] ->
        failwith "pss cannot be empty"
    | ps :: [] ->
        List.map (fun x -> [x]) (build_row ps n)
    | ps :: pss' ->
        let* next_row  = build_row ps n in
        let* cand_rows = build_candidate pss' n in
            [next_row :: cand_rows]

let verify_row ps xs = 
    let rec help xs count =
        match xs with
        | [] -> 
            if count = 0 then [] else [count]
        | false :: xs' -> 
            if count = 0 then help xs' 0 else count :: help xs' 0
        | true :: xs' -> 
            help xs' (count + 1)
    in
    ps = (help xs 0)
    
let rec verify_rows pss xss =
    match pss, xss with
    | [], _ | _, [] -> 
        true
    | ps :: pss', xs :: xss' ->
        (verify_row ps xs) && (verify_rows pss' xss')
            
let transpose xss = 
    let rec help xss res = 
        match xss with
        | [] -> 
            List.rev res
        | [] :: xss' -> 
            help [] res
        | (x :: xs) :: xss' -> 
            help (List.map List.tl xss) ((x :: (List.map List.hd xss')) :: res)
    in
    help xss []

type nonogram_spec = {rows: int list list; cols: int list list}

let solve_nonogram nono =
    build_candidate (nono.rows) (List.length (nono.cols))
    |> List.filter (fun xss -> transpose xss |> verify_rows nono.cols)

val ( let* ) : 'a list -> ('a -> 'b list) -> 'b list = <fun>


val create_list : int -> 'a -> 'a list = <fun>


val build_row : int list -> int -> bool list list = <fun>


val build_candidate : int list list -> int -> bool list list list = <fun>


val verify_row : int list -> bool list -> bool = <fun>


val verify_rows : int list list -> bool list list -> bool = <fun>


val transpose : 'a list list -> 'a list list = <fun>


type nonogram_spec = { rows : int list list; cols : int list list; }


val solve_nonogram : nonogram_spec -> bool list list list = <fun>


In [2]:
let example_1 = {
  rows = [[2];[1];[1]];
  cols = [[1;1];[2]]
};;


let example_2 = {
  rows = [[2];[2;1];[1;1];[2]];
  cols = [[2];[2;1];[1;1];[2]]
}

let big_example = {
  rows = [[1;2];[2];[1];[1];[2];[2;4];[2;6];[8];[1;1];[2;2]];
  cols = [[2];[3];[1];[2;1];[5];[4];[1;4;1];[1;5];[2;2];[2;1]]
}

val example_1 : nonogram_spec =
  {rows = [[2]; [1]; [1]]; cols = [[1; 1]; [2]]}


val example_2 : nonogram_spec =
  {rows = [[2]; [2; 1]; [1; 1]; [2]]; cols = [[2]; [2; 1]; [1; 1]; [2]]}


val big_example : nonogram_spec =
  {rows = [[1; 2]; [2]; [1]; [1]; [2]; [2; 4]; [2; 6]; [8]; [1; 1]; [2; 2]];
   cols =
    [[2]; [3]; [1]; [2; 1]; [5]; [4]; [1; 4; 1]; [1; 5]; [2; 2]; [2; 1]]}


In [3]:
build_row [2] 3

- : bool list list = [[true; true; false]; [false; true; true]]


In [4]:
solve_nonogram example_1

- : bool list list list = [[[true; true]; [false; true]; [true; false]]]


In [5]:
solve_nonogram example_2

- : bool list list list =
[[[false; true; true; false]; [true; true; false; true];
  [true; false; false; true]; [false; true; true; false]]]


In [6]:
two_num_product 6 12

error: compile_error

In [7]:
let* cand = build_candidate [[1]] 3 in
    cand

- : bool list list =
[[true; false; false]; [false; true; false]; [false; false; true]]


In [8]:
let* cand = build_row [1] 3 in
    [cand]

- : bool list list =
[[true; false; false]; [false; true; false]; [false; false; true]]


In [9]:
build_row [1] 3

- : bool list list =
[[true; false; false]; [false; true; false]; [false; false; true]]


In [10]:
transpose [[1;2];[3;4];[5;6]]

- : int list list = [[1; 3; 5]; [2; 4; 6]]


In [11]:
transpose [[1;2;3];[4;5;6]]

- : int list list = [[1; 4]; [2; 5]; [3; 6]]


In [12]:
verify_row [1;1] [false;true;false;false;true;]

- : bool = true


In [42]:
solve_nonogram big_example

interrupt: intterupt

In [13]:
List.length (build_row [1;2] 10)

- : int = 28


In [14]:
[[1;2];[2];[1];[1];[2];[2;4];[2;6];[8];[1;1];[2;2]]

- : int list list =
[[1; 2]; [2]; [1]; [1]; [2]; [2; 4]; [2; 6]; [8]; [1; 1]; [2; 2]]


In [15]:
List.length (build_row [2;2] 10)

- : int = 21


In [16]:
assert ((verify_row [1;2;3] [true;false;true;true;false;false;true;true;true]) = true);;
assert ((verify_row [1;1] [true;false;true;false;true]) = false)

- : unit = ()


- : unit = ()


In [17]:
build_row [1;2] 10

- : bool list list =
[[true; false; true; true; false; false; false; false; false; false];
 [true; false; false; true; true; false; false; false; false; false];
 [true; false; false; false; true; true; false; false; false; false];
 [true; false; false; false; false; true; true; false; false; false];
 [true; false; false; false; false; false; true; true; false; false];
 [true; false; false; false; false; false; false; true; true; false];
 [true; false; false; false; false; false; false; false; true; true];
 [false; true; false; true; true; false; false; false; false; false];
 [false; true; false; false; true; true; false; false; false; false];
 [false; true; false; false; false; true; true; false; false; false];
 [false; true; false; false; false; false; true; true; false; false];
 [false; true; false; false; false; false; false; true; true; false];
 [false; true; false; false; false; false; false; false; true; true];
 [false; false; true; false; true; true; false; false; false; false];

In [18]:
List.map (fun x -> List.length (build_row x 10)) [[1;2];[2];[1];[1];[2];[2;4];[2;6];[8];[1;1];[2;2]]

- : int list = [28; 9; 10; 10; 9; 10; 3; 3; 36; 21]
